# 추가 실습: 오류 점검과 기간 비교

`first-eda.ipynb` 다음에 해 볼 두 실험입니다. 같은 폴더의 `samsung-2024.csv`를 읽으며 원본 파일에는 쓰지 않습니다. 아래 코드 셀은 직접 실행하고 출력을 읽는 추가 실습입니다. 기존 실습과 같은 Python 커널에서 원하는 실험을 직접 실행하세요. 종가는 원, 거래량은 주입니다. 출처와 종가 계열의 범위는 [자료안내.md](자료안내.md)를 따릅니다.

## 실험 1. 의도적으로 만든 오류 연습 표

원본의 첫 세 행을 복사해 **두 번째 행의 Close만 빈값**으로 바꾸고, 첫 번째 행을 한 번 더 붙인 뒤 행 순서를 뒤집습니다. 아래 표는 실제 원본의 오류를 보여 주는 자료가 아닙니다. 인덱스는 원래 행을 추적하도록 유지합니다.

표, 열별 빈값 수, 중복 날짜 수, 중복 날짜에 해당하는 모든 행, 날짜 오름차순 여부를 출력합니다. 중복 날짜 수는 **같은 날짜의 첫 행을 제외한 추가 행 수**이며, 중복 행 표에는 첫 행도 포함합니다. 같은 날짜가 이어져도 오름차순일 수 있으므로 중복과 순서는 따로 점검합니다. 자동으로 행을 삭제하거나 빈값을 채우지 않습니다.

**남길 질문:** 실제 자료에서 이런 문제가 보인다면, 원본 CSV의 해당 날짜 행과 제공처의 일별 원자료·수집 기록 중 무엇을 더 확인해야 할까요? 행 전체가 빠졌는지는 거래소 거래일 달력과 어떻게 대조할까요?

**요청 예시:** “이 표는 의도적으로 만든 오류 연습 표입니다. 빈값·중복 날짜·순서 문제를 원래 인덱스와 함께 설명하고, 수정 전에 확인할 원래 자료를 질문으로 남겨 주세요. 자동으로 삭제하거나 채우지 마세요.”

In [1]:
import pandas as pd
from IPython.display import display

original = pd.read_csv("samsung-2024.csv", parse_dates=["Date"])
practice = original.head(3).copy()
# 종가의 정수 값과 빈값을 함께 담을 수 있게 준비합니다.
practice["Close"] = practice["Close"].astype("Int64")
practice.loc[practice.index[1], "Close"] = pd.NA
practice = pd.concat([practice, practice.iloc[[0]]]).iloc[::-1].copy()

print("의도적으로 만든 오류 연습 표")
display(practice)
print("열별 빈값 수")
display(practice.isna().sum().rename("빈값 수").to_frame())
print(f"중복 날짜 수 (첫 행 제외): {practice['Date'].duplicated().sum()}개")
print("중복 날짜에 해당하는 모든 행 (첫 행 포함)")
display(practice.loc[practice["Date"].duplicated(keep=False)])
order_ok = practice["Date"].is_monotonic_increasing
print(f"날짜가 오름차순인가요? {'예' if order_ok else '아니요'} ({order_ok})")

의도적으로 만든 오류 연습 표


,Date,Close,Volume
0,2024-01-02,79600,17142847
2,2024-01-04,76600,15324439
1,2024-01-03,<NA>,21753644
0,2024-01-02,79600,17142847


열별 빈값 수


,빈값 수
Date,0
Close,1
Volume,0


중복 날짜 수 (첫 행 제외): 1개
중복 날짜에 해당하는 모든 행 (첫 행 포함)


,Date,Close,Volume
0,2024-01-02,79600,17142847
0,2024-01-02,79600,17142847


날짜가 오름차순인가요? 아니요 (False)


## 실험 2. 1~6월과 7~12월 비교

원본 전체를 새로 읽고 2024년 **1월 1일 이상·7월 1일 미만**, **7월 1일 이상·2025년 1월 1일 미만**으로 필터합니다. 비교표의 각 행은 **원래 CSV에서 해당 기간을 필터한 결과**이며, 실험 1의 오류 연습 표는 사용하지 않습니다. 행 수는 각 기간에 저장된 거래일 수이고, 실제 시작·끝 날짜도 함께 표시합니다.

같은 표에서 종가 최댓값·최솟값(원), 하루 거래량 평균·중앙값(주)을 비교합니다. 평균은 각 기간의 거래량 합계를 해당 기간의 행 수로 나눈 값이며, 표에서는 평균·중앙값을 소수 첫째 자리까지 표시합니다. **기간에 따라 종가 범위와 대표 거래량에 대한 관찰은 달라질 수 있습니다. 수익률·원인·예측 계산은 하지 않았습니다.** 아직 실행하지 않았으므로 어느 기간이 더 큰지는 결과를 보고 적어 보세요.

**요청 예시:** “이 표의 두 행이 각각 원래 CSV의 어느 기간을 필터한 결과인지 확인해 주세요. 행 수·종가 범위·거래량 평균과 중앙값을 단위와 함께 비교하고, 한 종목의 두 기간에서 관찰한 사실만 적어 주세요. 수익률·원인·미래 예측은 계산하지 마세요.”

In [2]:
import pandas as pd
from IPython.display import display

original = pd.read_csv("samsung-2024.csv", parse_dates=["Date"])
periods = [
    ("2024년 1~6월", "2024-01-01", "2024-07-01"),
    ("2024년 7~12월", "2024-07-01", "2025-01-01"),
]
rows = []
for label, start, end in periods:
    # 각 요약 행은 원래 CSV에서 날짜로 필터한 결과입니다.
    period = original.loc[original["Date"].ge(start) & original["Date"].lt(end)]
    rows.append({
        "기간": label,
        "실제 첫 날짜": period["Date"].min(),
        "실제 마지막 날짜": period["Date"].max(),
        "행 수": len(period),
        "종가 최댓값(원)": period["Close"].max(),
        "종가 최솟값(원)": period["Close"].min(),
        "평균 거래량(주)": period["Volume"].mean(),
        "중앙 거래량(주)": period["Volume"].median(),
    })

comparison = pd.DataFrame(rows)
print("각 행은 원래 CSV에서 해당 기간을 필터한 요약입니다.")
display(comparison.style.hide(axis="index").format({
    "실제 첫 날짜": "{:%Y-%m-%d}",
    "실제 마지막 날짜": "{:%Y-%m-%d}",
    "행 수": "{:,.0f}",
    "종가 최댓값(원)": "{:,.0f}",
    "종가 최솟값(원)": "{:,.0f}",
    "평균 거래량(주)": "{:,.1f}",
    "중앙 거래량(주)": "{:,.1f}",
}))

각 행은 원래 CSV에서 해당 기간을 필터한 요약입니다.


기간,실제 첫 날짜,실제 마지막 날짜,행 수,종가 최댓값(원),종가 최솟값(원),평균 거래량(주),중앙 거래량(주)
2024년 1~6월,2024-01-02,2024-06-28,121,"85,300","71,000","19,853,578.7","18,717,699.0"
2024년 7~12월,2024-07-01,2024-12-30,123,"87,800","49,900","23,592,977.3","22,092,218.0"
